#### IMPORT ALL THE NEEDED LIBRARY FOR THE INDEXING

In [2]:
import pandas as pd
import ast
import numpy as np
import faiss

#### LOAD IN THE DATASETS WITH EMBEDDING

In [3]:
df = pd.read_csv("tickets_with_embeddings.csv")
embeddings = np.vstack(df['embedding'].apply(ast.literal_eval).values).astype("float32")

#### NORMALIZE AND INDEX

In [4]:
faiss.normalize_L2(embeddings)

# Build FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)   # inner product = cosine
index.add(embeddings)

#### SEARCH FOR DUPES (TOP 5)

In [5]:
threshold = 0.95
k = 5

to_drop = set()

for i in range(len(embeddings)):
    if i in to_drop:
        continue

    sims, idxs = index.search(embeddings[i].reshape(1, -1), k)

    for sim, j in zip(sims[0], idxs[0]):
        if j == i:
            continue
        if sim > threshold:
            to_drop.add(j)

#### DROP THE DUPES

In [6]:
dedup_df = df.drop(index=list(to_drop)).reset_index(drop=True)

print(f"Removed {len(to_drop)} near-duplicates")
print(f"Remaining rows: {len(dedup_df)}")

dedup_df.to_csv("tickets_deduped.csv", index=False)

Removed 1953 near-duplicates
Remaining rows: 25649


#### FINAL INDEXING

In [7]:
dedup_embeddings = np.vstack(dedup_df['embedding'].apply(ast.literal_eval).values).astype("float32")
faiss.normalize_L2(dedup_embeddings)

# Build FAISS index
dedup_dim = dedup_embeddings.shape[1]
index = faiss.IndexFlatIP(dedup_dim)   # inner product = cosine
index.add(dedup_embeddings)

print("Vectors indexed:", index.ntotal)

# Save index
faiss.write_index(index, "tickets.faiss")

Vectors indexed: 25649
